In [1]:
import re
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
import os
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = False

In [2]:
def pdf2md_single(source) :
    converter = DocumentConverter(format_options={ InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, backend=PyPdfiumDocumentBackend
            )
        }
    )
    result = converter.convert(source)
    return result.document.export_to_markdown()


#file = open('pdf2md.md','w',encoding='utf-8')
#file.write(pdf2md_single("../unique_sxolika_pdf/paper_16.pdf"))

In [29]:
def paragraph_maker(text,maxpadding = 2) :
    lines = text.splitlines()
    paragraphs = []
    emptyline = 0
    paragraph = ''
    for i,line in enumerate(lines) :
        if i == len(lines) - 1 :
            paragraph = paragraph + line
            paragraphs.append(paragraph)
            continue
        if line == '' :
            emptyline = emptyline + 1
            if emptyline == maxpadding :
                if paragraph != '' :
                    paragraphs.append(paragraph)
                emptyline = 0
                paragraph = ''
        else :
            paragraph = paragraph + line + '\n'
            emptyline = 0
            if i == len(lines) - 1 :
                paragraphs.append(paragraph)
    return paragraphs

def paragraph_merger(paragraphs,min_par_size,threshold) :
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if len(paragraph) < min_par_size :
            if paragraph != paragraphs[-1] :
                if len(paragraphs[i+1]) > threshold :
                    if not (paragraphs[i+1].startswith('##') or paragraphs[i+1].startswith('|') ) :
                        paragraphs[i+1] = paragraph + '\n MERGE \n' + paragraphs[i+1]
                        continue
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_image(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('<!-- image -->') :
            continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_dotlines(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('.....') :
            if paragraph.endswith('.....') :
                continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_remove_artifacts(paragraphs,min_length = 5) :
    newparagraphs = []
    for paragraph in paragraphs :
        if len(paragraph) <= min_length :
            continue
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_not_char_end(paragraph,chars) :
    #if not paragraph.startswith('##') :
    end_search_flag = False
    if not paragraph.endswith(chars) :
        paragraph = '[Out:Paragraph does not end with specified char]' + paragraph
        end_search_flag = True
    return (paragraph,end_search_flag)

def all_paragraph_not_char_end(paragraphs,chars) :
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        newparagraphs.append(paragraph_not_char_end(paragraph,chars)[0])
        if paragraph_not_char_end(paragraph,chars)[1] == False :
            for t_paragraph in paragraphs[i+1:] :
                newparagraphs.append(t_paragraph)
            return newparagraphs
    return newparagraphs

def paragraph_fix_broken_line(paragraphs) :
    ending_lower_pattern = re.compile(r'(.*[α-ωά-ώa-zΑ-ΩΆ-Ώ]\n)|(.*[0-9]\n)|(.*°C\n)|(.*\)\n)|(.*\-\n)|(.*,\n)')
    starting_lower_pattern = re.compile(r'([α-ωά-ώa-z])|([0-9])|(°)|\)|- ')
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if re.match(ending_lower_pattern,paragraph) :
            if paragraph != paragraphs[-1] :
                if re.match(starting_lower_pattern,paragraphs[i+1]) :
                    paragraphs[i+1] = paragraph + 'MERGE\n' + paragraphs[i+1]
                    continue
        newparagraphs.append(paragraph)
    return newparagraphs

def print_text(paragraphs) :
    for paragraph in paragraphs :
        print(paragraph)
        print('\n\n\n')

def remove_ending_chunk(paragraphs) :
    paragraphs = paragraphs[::-1]
    for i,paragraph in enumerate(paragraphs) :
        if paragraph.startswith('##') :
            paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
            return paragraphs[::-1]
        paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
        #del paragraphs[-1]
    return paragraphs[::-1]

def remove_funding(paragraphs) :
    funding_pattern = re.compile(r'συγχρηματοδοτείται από την Ελλάδα και την Ευρωπαϊκή Ένωση|συγχρηµατοδοτείται από την Ελλάδα και την Ευρωπαϊκή Ένωση')
    newparagraphs = []
    for paragraph in paragraphs :
        if re.search(funding_pattern,paragraph) :
            continue
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_bibliography(paragraphs,bibliography) :
    newparagraphs = []
    bibliography_flag = False
    for paragraph in paragraphs :
        if paragraph.startswith(bibliography) :
            bibliography_flag = True
            paragraph = '[Out:Paragraph is Bibliography]' + paragraph
        elif bibliography_flag == True :
            if paragraph.startswith('##') :
                bibliography_flag = False
                continue
            paragraph = '[Out:Paragraph is Bibliography]' + paragraph
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_legal_note_3966_2011(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('Βάσει του ν. 3966/2011') :
            paragraph = '[Out:Legal note 3966/2011]' + paragraph
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_glossary(paragraphs,glossary_tags) :
    newparagraphs = []
    glossary_flag = False
    for paragraph in paragraphs :
        if paragraph.startswith(glossary_tags) :
            glossary_flag = True
            paragraph = '[Out:Paragraph is Glossary]' + paragraph
        elif glossary_flag == True :
            if paragraph.startswith('##') :
                glossary_flag = False
                continue
            paragraph = '[Out:Paragraph is Glossary]' + paragraph
        newparagraphs.append(paragraph)
    return newparagraphs

def write_text(paragraphs,file) :
    text = '[Paragraph Begin: Number 0 ]'
    for i,paragraph in enumerate(paragraphs) :
        text = text + paragraph + '[Paragraph End]\n\n\n[Paragraph Begin: Number ' + str(i+1) + ' ]'
    file.write(text) 

In [30]:
os.makedirs('../test_output', exist_ok=True)
for i,file in enumerate(os.listdir('../md_output')) :
    if not file.endswith('.md'):
            continue
    try:
        with open('../md_output'+'/'+file, 'r', encoding='utf-8') as infile:
            text = infile.read()
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    endings = ('.\n','?\n','!\n',';\n','.\n',';\n','|\n')
    bibliography_tags = ('## ΞΕΝΟΓΛΩΣΣΗ','## ΕΛΛΗΝΙΚΗ','## Bl ΒΛΙΟΓΡΑΦΙΑ', '## ΒΛΙΟΓΡΑΦΙΑ','## Π Ι ΝΑ Κ Α Σ Π Ρ Ο Ε Λ Ε Υ Σ Η Σ Τ Ω Ν E Ι ΚΟ Ν Ω Ν','## ΠΙΝΑΚΑΣ ΠΡΟΕΛΕΥΣΗΣ ΤΩΝ EΙΚΟΝΩΝ','## ΒΙΒΛΙΟΓΡΑΦΙΑ')
    glossary_tags = ('## ΓΛΩΣΣΑΡΙ, ## Γλωσσάρι','## Λ EΞIKO','## ΛEΞIKO','## ΛΕΞΙΛΟΓΙΟ')

    paragraphs = paragraph_maker(text,maxpadding=1)
    paragraphs = paragraph_clean_image(paragraphs)
    paragraphs = paragraph_clean_dotlines(paragraphs)
    paragraphs = paragraph_remove_artifacts(paragraphs)
    paragraphs = paragraph_fix_broken_line(paragraphs)
    paragraphs = remove_funding(paragraphs)
    paragraphs = remove_legal_note_3966_2011(paragraphs)
    paragraphs = paragraph_merger(paragraphs,500,10)
    paragraphs = remove_bibliography(paragraphs,bibliography_tags)
    paragraphs = remove_glossary(paragraphs,glossary_tags)
    paragraphs = all_paragraph_not_char_end(paragraphs,endings)
    #paragraphs = remove_ending_chunk(paragraphs)
    
    t_file = 'test_'+file

    output_file_path = os.path.join('../test_output', t_file)
    try:
        with open(output_file_path, 'w', encoding='utf-8') as output_file:
            write_text(paragraphs,output_file)
    except Exception as e:
            print(f"Error writing to {output_file_path}: {e}")
            continue